# Notebook overview
Import the libraries used to load catalogues, prepare image payloads, and upload subjects to Panoptes.

In [237]:
from collections.abc import Sequence
import tqdm
import os
import io
import glob
import pandas as pd
import numpy as np
from typing import Any, TypeAlias, cast
from panoptes_client import Panoptes, Subject, SubjectSet, Project
from panoptes_client.panoptes import PanoptesAPIException
import getpass
import humanize
import tarfile
from PIL import Image

UploadBuffer: TypeAlias = dict[str, Any]
ManifestEntry: TypeAlias = dict[str, Any]
UploadImage: TypeAlias = UploadBuffer | str

## Paths and filenames
Define the data directories, catalogue filenames, prediction file, and image handling mode used by the rest of the notebook.

In [238]:
DATA_DIR = "/Users/hjd229/Documents/Data/Clump_Scout_2"
IMAGE_DIR = "subject_images"
MANIFEST_DIR = "subject_manifests"
PREDICTIONS_DIR = "FRCNN_predictions"
CATALOGUES_DIR = "catalogues"
PREDICTIONS_FILE = "preds_Euclid_nms.gzip"
MASTER_CATALOGUE_FILE = "_master_catalog.csv"
MORPH_CATALOGUE_FILE = "morphology_catalogue.parquet"
MAX_UPLOAD_SIZE = 1000 * 1000
# Set this to True if you want to read data directly from tarballs
USING_TARFILES = True

## Load source tables
Read the predictions table and both catalogues from disk so they can be filtered and joined before upload.

In [239]:
# Load the detector output and the catalogue tables used to decide which objects to upload.
predictions = pd.read_parquet(os.path.join(DATA_DIR, PREDICTIONS_DIR, PREDICTIONS_FILE))
master_catalogue = pd.read_csv(
    os.path.join(DATA_DIR, CATALOGUES_DIR, MASTER_CATALOGUE_FILE)
)
morph_catalogue = pd.read_parquet(
    os.path.join(DATA_DIR, CATALOGUES_DIR, MORPH_CATALOGUE_FILE)
).astype({"object_id": int})

## Combine catalogue data
Join the master and morphology catalogues on `object_id` so the selection step can use fields from both tables.

In [240]:
combined_catalogue = master_catalogue.merge(
    morph_catalogue, left_on="object_id", right_on="object_id", how="inner"
)

## Select upload candidates
Filter for galaxies that are large enough and likely to be featured disks, then keep the matching prediction IDs.

In [241]:
# select galaxies that aren't just smooth and are large enough to be classifiable
selector = (combined_catalogue.segmentation_area_y >= 2000) & (
    combined_catalogue["smooth-or-featured_featured-or-disk_fraction"] > 0.5
)
selected_ids = predictions.local_ids[
    predictions.local_ids.isin(combined_catalogue[selector].object_id)
].drop_duplicates()

## Filename helpers
Convert between integer object IDs and the filename tokens used inside the image tarballs, including negative IDs.

In [242]:
def id_to_filename_pattern(id: int) -> str:
    # Tarball filenames encode negative IDs with a NEG prefix instead of a minus sign.
    if id < 0:
        return f"NEG{abs(id)}"
    return str(id)


def filename_pattern_to_id(pattern: str) -> int:
    # Convert the filename token back into the integer ID used in the catalogues.
    if pattern.startswith("NEG"):
        return -int(pattern[3:])
    try:
        return int(pattern)
    except ValueError as e:
        print(e, pattern)
        return -1

## Subject-set linking helper
Attach a batch of uploaded subjects to a subject set and return the subject IDs with their filenames for manifest generation.

In [243]:
def link_subject_set(
    client: Panoptes, subject_set: SubjectSet, subjects: list[Subject]
) -> list[ManifestEntry]:
    # Subjects are uploaded first and then attached to the target subject set in batches.
    with client:
        subject_set.add(subjects)
        subject_set.save()

    return [
        {"subject_id": subject.id, "filename": subject.metadata["filename"]}
        for subject in subjects
    ]

## Authenticate with Panoptes
Prompt for Zooniverse credentials and open a client connection that the upload functions reuse.

In [244]:
username = getpass.getpass("Panoptes username: ")
password = getpass.getpass("Panoptes password: ")

client = Panoptes.connect(username=username, password=password)

PanoptesAPIException: Invalid email or password.

## Subject upload routine
Create or reuse a subject set, validate image sizes, upload each subject image bundle, and write a manifest of uploaded subject IDs.

In [ ]:
def upload(
    subject_images: Sequence[Sequence[UploadImage]],
    project_id: int,
    output_manifest: str,
    subject_set_name: str | None,
    subject_set_id: int | None = None,
) -> None:

    if (subject_set_name is None) and (subject_set_id is None):
        raise ValueError(
            "Please enter a subject set name to create a new subject set or a subject set ID to upload to an existing set"
        )

    # Resolve the destination project and subject set once before uploading any images.
    project = Project(project_id)

    if subject_set_name is not None:
        subject_set = SubjectSet()
        subject_set.links.project = project
        subject_set.display_name = subject_set_name
        subject_set.save()
    else:
        subject_set = SubjectSet(subject_set_id)

    manifest: list[ManifestEntry] = []
    subjects: list[Subject] = []
    for images in tqdm.tqdm(
        subject_images, ascii=True, dynamic_ncols=True, desc="Creating subjects"
    ):
        # Panoptes enforces a per-file upload limit, so validate each panel before creating the subject.
        if USING_TARFILES:
            for image in images:
                tar_image = cast(UploadBuffer, image)
                location = cast(io.BytesIO, tar_image.get("location"))
                size_in_bytes = location.getbuffer().nbytes
                if size_in_bytes > MAX_UPLOAD_SIZE:
                    raise PanoptesAPIException(
                        f"{tar_image} has file size {humanize.naturalsize(size_in_bytes)} which is greater than the upload limit of {humanize.naturalsize(MAX_UPLOAD_SIZE)}"
                    )
            # Keep the original basenames for the manifest even when the file content comes from in-memory buffers.
            image_file_names = [
                os.path.basename(str(cast(UploadBuffer, image).pop("filename")))
                for image in images
            ]
        else:
            for image in images:
                file_path = cast(str, image)
                if os.path.getsize(file_path) > MAX_UPLOAD_SIZE:
                    raise PanoptesAPIException(
                        f"{file_path} has file size {humanize.naturalsize(os.path.getsize(file_path))} which is greater than the upload limit of {humanize.naturalsize(MAX_UPLOAD_SIZE)}"
                    )
            image_file_names = [os.path.basename(cast(str, image)) for image in images]

        subject = Subject()
        subject.links.project = subject_set.links.project
        for image, image_file_name in zip(images, image_file_names):
            if USING_TARFILES:
                subject.add_location(**cast(UploadBuffer, image))
            else:
                subject.add_location(cast(str, image))
        # Store a representative filename so downstream manifests can map annotations back to the source image set.
        subject.metadata.update({"filename": image_file_name})
        subject.save()

        subjects.append(subject)

        # Link subjects in chunks to avoid keeping an arbitrarily large unsaved batch in memory.
        if len(subjects) > 100:
            manifest.extend(link_subject_set(client, subject_set, subjects))
            subjects.clear()

    # Flush the final partial batch and persist the subject-to-filename mapping for later Caesar imports.
    manifest.extend(link_subject_set(client, subject_set, subjects))

    manifest_df = pd.DataFrame.from_records(manifest)
    manifest_df.to_csv(output_manifest)

## Discover image variants
Build the list of available image variants from tar files or directories, depending on the configured storage format.

In [ ]:
if USING_TARFILES:
    image_variants = np.array(list(
        map(
            lambda x: os.path.basename(x).replace(".tar", ""),
            sorted(glob.glob(os.path.join(DATA_DIR, IMAGE_DIR, "*.tar"))),
        ))
    )[[1,2,0]] # This gives the same image order as in the beta.
else:  # using untarred directories
    image_variants = list(
        map(os.path.basename, glob.glob(os.path.join(DATA_DIR, IMAGE_DIR, "*")))
    )[-1::-1]

## Open a tarball for one variant
Load a variant tarball, list the JPEG members it contains, and map each embedded filename back to its object ID.

In [ ]:
def process_tar_for_variant(variant: str) -> tuple[tarfile.TarFile, list[str], list[int]]:
    tarfile_object = tarfile.open(os.path.join(DATA_DIR, IMAGE_DIR, f"{variant}.tar"))
    tarfile_image_filenames = list(
        filter(lambda x: x.endswith(".jpg"), tarfile_object.getnames())
    )
    # Each filename embeds the object ID in the second underscore-delimited field.
    tarfile_objids = [
        filename_pattern_to_id(name.split("_")[1]) for name in tarfile_image_filenames
    ]
    # check that all images are available
    assert selected_ids.isin(tarfile_objids).unique().squeeze()
    return tarfile_object, tarfile_image_filenames, tarfile_objids

## Resize extracted images
Normalize each extracted image to 400 by 400 pixels before it is sent to Panoptes.

In [ ]:
def resize_image(image_data: io.BytesIO) -> io.BytesIO:
    image = Image.open(image_data)
    # Resize every panel to a common display size before packaging it for upload.
    resized = image.resize((400, 400))
    output = io.BytesIO()
    resized.save(output, format="JPEG")
    return output

## Extract one subject image
Read a single object image from a tarball, resize it, rewind the buffer, and package it in the format expected by `subject.add_location()`.

In [ ]:
def extract_image_for_id(
    id: int,
    tarfile_object: tarfile.TarFile,
    tarfile_image_filenames: list[str],
    tarfile_objids: list[int],
) -> UploadBuffer:
    # Look up the tarball member by object ID so every variant uses the same target ordering.
    filename = tarfile_image_filenames[tarfile_objids.index(id)]
    extracted_file = tarfile_object.extractfile(filename)
    if extracted_file is None:
        raise FileNotFoundError(f"Could not extract {filename} from tarfile")
    image_bytesio = resize_image(
        io.BytesIO(extracted_file.read())
    )
    # Rewind the buffer because Panoptes reads from the current stream position.
    image_bytesio.seek(0)
    return dict(location=image_bytesio, manual_mimetype="image/jpeg", filename=filename)


# subject.add_location(io.BytesIO(extracted.read()), manual_mimetype="image/jpeg")

## Build the image lookup
Create the in-memory image sources used for upload, either by opening tarballs once per variant or by listing untarred image files.

In [ ]:
if USING_TARFILES:
    # Open each tarball once and reuse the handles while subjects are assembled.
    images = {variant: process_tar_for_variant(variant) for variant in image_variants}
else:
    images = {
        variant: sorted(
            glob.glob(os.path.join(DATA_DIR, IMAGE_DIR, variant, "*424x424.jpg"))
        )
        for variant in image_variants
    }

    num_images = [len(image_variant_paths) for image_variant_paths in images.values()]

    numbers_match = len(set(num_images)) == 1

    assert numbers_match

invalid literal for int() with base 10: 'checkpoints/102018212' checkpoints/102018212
invalid literal for int() with base 10: 'checkpoints/102018212' checkpoints/102018212


## Validate untarred variant alignment
When working from directories instead of tar files, check that corresponding images across variants refer to the same objects.

In [ ]:
if not USING_TARFILES:
    variants_match = np.all(
        [
            len(
                set(
                    map(
                        lambda x: "".join(os.path.basename(x).split("_")[:2]),
                        image_variant_paths,
                    )
                )
            )
            == 1
            for image_variant_paths in zip(*images.values())
        ]
    )
    assert variants_match

## Upload subject batches
Split the selected targets into batches, assemble the image payload for each subject, and upload each batch to a new subject set.

In [ ]:
sset_size = 200
project_id = 22358
if USING_TARFILES:
    # Limit the first run while testing tarfile-backed uploads.
    max_num_images = 200
else:
    max_num_images = num_images[0]
for start in range(0, max_num_images, sset_size):
    end = min(start + sset_size, max_num_images)
    subject_set_name = f"test_subjects_{start}_{end}"
    output_manifest = os.path.join(DATA_DIR, MANIFEST_DIR, f"{subject_set_name}.csv")
    if USING_TARFILES:
        # Build one multi-panel subject payload per selected object ID.
        image_paths = [
            [
                extract_image_for_id(selected_ids.iloc[i], *cast(tuple[tarfile.TarFile, list[str], list[int]], v))
                for v in images.values()
            ]
            for i in range(start, end)
        ]
    else:
        image_paths = [
            [cast(list[str], v)[i] for v in images.values()] for i in range(start, end)
        ]
    upload(
        image_paths, project_id, output_manifest, subject_set_name
    ) 

Creating subjects: 100%|##########| 200/200 [02:43<00:00,  1.22it/s]


## Rebuild a manifest from Panoptes
Fetch an existing subject set and reconstruct a subject ID to filename manifest if the original upload manifest is missing or incomplete.

In [ ]:
# if something goes wrong in the subject upload use this function to generate a manifest
# that can be used to update annotations to Caesar
def generate_manifest(subject_set_id: int, filename: str) -> pd.DataFrame:
    ss = SubjectSet.find(subject_set_id)
    if ss is None:
        raise ValueError(f"No subject set found for id {subject_set_id}")
    manifest = pd.DataFrame(
        [[s.id, s.metadata["filename"]] for s in ss.subjects],
        columns=["subject_id", "filename"],
    )
    manifest.to_csv(filename)
    return manifest

## Example manifest recovery
Generate a manifest CSV for a known subject set ID and write it into the manifest directory.

In [79]:
subject_set_name = f"beta_subjects_0_200"
output_manifest = os.path.join(DATA_DIR, MANIFEST_DIR, f"{subject_set_name}.csv")
generate_manifest(129271, output_manifest)

,subject_id,filename
0,110778242,102019129_NEG615744122503813630_gz_arcsinh_vis...
1,110778243,102019129_NEG617313094507254999_gz_arcsinh_vis...
2,110778244,102019130_NEG618417628503412733_gz_arcsinh_vis...
3,110778245,102019130_NEG619635069502693923_gz_arcsinh_vis...
4,110778246,102019130_NEG621411534503511584_gz_arcsinh_vis...
...,...,...
195,110778337,102020055_NEG561864015497288472_gz_arcsinh_vis...
196,110778338,102020055_NEG563068192495450870_gz_arcsinh_vis...
197,110778339,102020055_NEG563139508497427598_gz_arcsinh_vis...
198,110778340,102020055_NEG563486252496793337_gz_arcsinh_vis...


## Scratch cell
Leave this final cell empty for ad hoc checks or one-off inspection commands.